In [ ]:
"""
Step 02: Timestamp Indicator (TSIND) Data Collection

Objective:

    - One row per minute (43 fields + date): raw bar, price-action fields, candle shape, stochastic, MACD, RSI,
      Bollinger Bands, Donchian channel and market structure, each computed on the snapshot that ends at that minute.
    - The forward-shifted price-action fields are read from the second-to-last snapshot row (the last one is NaN).
    - Logic unchanged from the step06_TSIND_data_collection notebook of the previous workspace (snapshot_features.get_TSIND_dict).
      Existing TSIND_data_YYYYMMDD.csv files can be copied into step02_TSIND_data instead of being regenerated.

Prerequisite: step01 (the snapshots are read from the cache, or created and cached when missing).
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT PLOTLY
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION
from so import config
# IMPORT TRANSFORM OHLCV DATA FUNCTIONS
from so.features.ohlcv_data_utils import format_ohlcv_pdf
# IMPORT SNAPSHOT FEATURE FUNCTIONS
from so.features.snapshot_features import get_date_TSIND_pdf
# IMPORT PLOT OHLCV FEATURES FUNCTIONS
from so.features.plot_ohlcv_utils import fig_add_ohlcv
# IMPORT MARKET DATETIME FUNCTIONS
from so.core.datetime_utils import ny_tz
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import get_path_file_list, \
                                    read_pdf_from_csv_file_path_list, \
                                    read_date_range_csv_files_from_path, \
                                    generate_func_data_date_list, \
                                    get_missing_date_list
# IMPORT PATHS
from so import paths

In [ ]:
# CALL FUNCTION TO GET ALL FILES IN PATH
rawzone_data_file_path_list = get_path_file_list(paths.LOCAL_OHLCV_DATA_FILE_PATH_STR)[:]
# CALL FUNCTION TO READ PDF FROM PATH LIST
raw_ohlcv_pdf = read_pdf_from_csv_file_path_list(rawzone_data_file_path_list)
# CORRECT THE TIMESTAMP DATE COLUMN
raw_ohlcv_pdf["timestamp"] = pd.to_datetime(raw_ohlcv_pdf.timestamp, utc=True).dt.tz_convert(ny_tz)
raw_ohlcv_pdf["date"] = raw_ohlcv_pdf["timestamp"].dt.date
# CALL FUNCTION TO COLLECT DATA AND SORT BY TIMESTAMP
complete_ohlcv_pdf = format_ohlcv_pdf(raw_ohlcv_pdf.copy()).sort_values("timestamp").reset_index(drop=True)
# DISPLAY INFORMATION
print(f"min timestamp:\t{complete_ohlcv_pdf.timestamp.min()}")
print(f"max timestamp:\t{complete_ohlcv_pdf.timestamp.max()}")
# PREVIEW DATAFRAME
complete_ohlcv_pdf

In [ ]:
# FUNCTION: GENERATE THE DATE TIMESTAMP INDICATOR (TSIND) DATAFRAME
def get_date_TSIND_data_pdf(date_str_in):
    # CALL FUNCTION TO GENERATE THE DATE TSIND DATAFRAME (READS OR CREATES THE STEP01 SNAPSHOTS)
    return get_date_TSIND_pdf(complete_ohlcv_pdf, date_str_in)

# DEFINE SAMPLE DATE
sample_date_str = str(complete_ohlcv_pdf["date"].iloc[-1])
# CALL FUNCTION TO GENERATE THE SAMPLE DATE TSIND DATAFRAME
sample_date_TSIND_pdf = get_date_TSIND_data_pdf(sample_date_str)
# DISPLAY INFORMATION
print(f"Item Count:\t{sample_date_TSIND_pdf.shape[1]}")
# PREVIEW DATAFRAME
sample_date_TSIND_pdf

In [ ]:
# FUNCTION: PLOT DATE TIMESTAMP INDICATOR (TSIND) DATAFRAME
def plot_TSIND_features_pdf(TSIND_pdf_in, width_in=1100, height_in=600):
    # CREATE A FIGURE
    fig = go.Figure()
    # ADD CANDLESTICK CHART
    fig_add_ohlcv(fig, TSIND_pdf_in)
    # ITERATE OVER THE LEVEL COLUMNS
    for col_str, mode_str in [("current_minima", "markers"), ("current_maxima", "markers"), ("bb_bot", "lines"), ("bb_mid", "lines"),
                              ("bb_top", "lines"), ("dc_bot", "lines"), ("dc_mid", "lines"), ("dc_top", "lines")]:
        # ADD THE LEVEL
        fig.add_trace(go.Scatter(x=TSIND_pdf_in["timestamp"], y=TSIND_pdf_in[col_str], mode=mode_str, name=col_str))
    # UPDATE LAYOUT
    fig.update_layout(title="Timestamp Indicator (TSIND) Features", yaxis_title="Price", xaxis_title="Time", showlegend=True,
                      xaxis_rangeslider_visible=False, template="plotly_dark", width=width_in, height=height_in, font=dict(color="white"))
    # DISPLAY FIGURE
    fig.show()

# CALL FUNCTION TO PLOT THE SAMPLE DATE TSIND FEATURES
plot_TSIND_features_pdf(sample_date_TSIND_pdf)

In [ ]:
# COLLECT THE START AND END DATE
start_date_str, end_date_str = str(complete_ohlcv_pdf["date"].min()), str(complete_ohlcv_pdf["date"].max())
# GET MISSING DATE LIST
missing_date_list = get_missing_date_list(paths.LOCAL_TSIND_DATA_FILE_PATH_STR, start_date_str, end_date_str)
# PREVIEW LIST
missing_date_list[:10]

In [ ]:
# CALL FUNCTION TO PROCESS DATE LIST
generate_func_data_date_list(get_date_TSIND_data_pdf, missing_date_list, paths.LOCAL_TSIND_DATA_FILE_PATH_STR, "TSIND_data", "I")

In [ ]:
# CALL FUNCTION TO READ THE LAST MONTH OF TSIND DATA
TSIND_data_pdf = read_date_range_csv_files_from_path(paths.LOCAL_TSIND_DATA_FILE_PATH_STR, str((pd.to_datetime(end_date_str) - pd.DateOffset(months=1)).date()), end_date_str)
# PREVIEW DATAFRAME
TSIND_data_pdf

In [ ]:
"""
Verification: the TSIND files were COPIED from the previous workspace (old step06), not regenerated. This cell regenerates 5
random dates with the current code and compares them with the stored files (CSV round trip, string compare).
Result 2026-10-04: 5 of 5 identical (docs/RESULTS_LOG.md).
"""
# IMPORT THE HELPERS
import io, random
# PICK 5 RANDOM DATES (FIXED SEED)
check_date_list = random.Random(0).sample(sorted(complete_ohlcv_pdf["date"].unique()), 5)
# ITERATE OVER THE DATES
for check_date in check_date_list:
    # REGENERATE THE DATE WITH THE CURRENT CODE (CSV ROUND TRIP)
    new_pdf = pd.read_csv(io.StringIO(get_date_TSIND_data_pdf(str(check_date)).to_csv(index=False)))
    # READ THE STORED FILE
    old_pdf = pd.read_csv(f"{paths.LOCAL_TSIND_DATA_FILE_PATH_STR}TSIND_data_{check_date:%Y%m%d}.csv").drop(columns=["date"], errors="ignore")
    # COMPARE THE COLUMNS AND THE VALUES
    same_bool = list(old_pdf.columns) == list(new_pdf.columns) and old_pdf.astype(str).equals(new_pdf.astype(str))
    # DISPLAY INFORMATION
    print(f"{check_date}\t{'✅ identical' if same_bool else '❌ differs'}")